In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# importing featured engineer train and test data

X_train_fe = pd.read_csv('../Dataset/X_train_fe.csv')
X_test_fe = pd.read_csv('../Dataset/X_test_fe.csv')

In [ ]:
# building a simple fe model

from sklearn.ensemble import RandomForestRegressor

fe_model = RandomForestRegressor(random_state=42)
fe_model.fit(X_train_fe, y_train)
fe_model_pred = fe_model.predict(X_test_fe)

In [ ]:
# testing fe model

from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

fe_result = pd.DataFrame({
    'R² Score':[r2_score(y_test, fe_model_pred)], # How much of the quality score can my model explain?
    'MAE Score': [mean_absolute_error(y_test, fe_model_pred)], # On average, the prediction is off by about half a quality point
    'RMSE Score': [root_mean_squared_error(y_test, fe_model_pred)] # some predictions have larger errors.
},
index=['Feature Engineering Model']) 


fe_result

In [ ]:
# Compare fe model and baseline model result


bs_result = pd.read_csv(
    '../Report/Results/baseline_results.csv',
    index_col=0
)


comparison_df = pd.concat(
    [bs_result, fe_result],
    ignore_index=False
)

comparison_df

In [ ]:
# try fe with cross validate

from sklearn.model_selection import cross_validate

rf = RandomForestRegressor(random_state=42)

cv_scores = cross_validate(
    rf,
    X_train_fe,
    y_train,
    cv=5,
    scoring={
        'r2': 'r2',
        'mae': 'neg_mean_absolute_error',
        'rmse': 'neg_root_mean_squared_error'
    }
)


In [ ]:
fe_cv_result = pd.DataFrame({
    'R² Score': [cv_scores['test_r2'].mean()],
    'MAE Score': [-cv_scores['test_mae'].mean()],
    'RMSE Score': [-cv_scores['test_rmse'].mean()]
}, index=['Feature Engineered CV'])

fe_cv_result

In [ ]:
# compared all features to check if fe is needed

bs_result = pd.read_csv(
    '../Report/Results/baseline_results.csv',
    index_col=0
)

total_comparison_df = pd.concat([
    bs_result,
    fe_result,
    fe_cv_result
], ignore_index=False)

total_comparison_df

In [ ]:
# checking the importance of fe columns

importance = pd.DataFrame({
    'Feature': X_train_fe.columns,
    'Importance': fe_model.feature_importances_
})

importance = importance.sort_values('Importance', ascending=False)
importance.head(15)

In [ ]:
# dropping the least important fe columns to test if it improves generalization

X_train_fe = X_train_fe.drop(['total_acidity', 'alcohol_acidity', 'extra_sulphur_dioxide'], axis=1)
X_test_fe = X_test_fe.drop(['total_acidity', 'alcohol_acidity', 'extra_sulphur_dioxide'], axis=1)

In [ ]:
rf_selected = RandomForestRegressor(
    random_state=42
)

rf_selected.fit(X_train_fe, y_train)
selected_pred = rf_selected.predict(X_test_fe)

In [ ]:
selected_result = pd.DataFrame({
    'R² Score': [r2_score(y_test, selected_pred)],
    'MAE Score': [mean_absolute_error(y_test, selected_pred)],
    'RMSE Score': [root_mean_squared_error(y_test, selected_pred)]
}, index=['Selected FE Model'])

selected_result

In [ ]:
comparison_df = pd.concat([
    bs_result,
    fe_result,
    selected_result
])

comparison_df